# Experimento 03 — Selección de modelos robusta al shift temporal

Este cuaderno repite el pipeline enriquecido del experimento 02 (baseline + distancia geográfica) pero cambia el criterio con el que elegimos campeón: en lugar de la media del score entre los tres pliegues de la validación cruzada temporal usamos el *peor* pliegue. La intuición, prestada de la literatura sobre CV para series temporales (Bergmeir & Benítez, 2012) y del aviso de Varma & Simon (2006) sobre tomar la media como garantía de generalización, es que un modelo cuyo peor pliegue es aceptable debería generalizar mejor cuando la distribución se mueve en el tiempo.

La configuración, los datos y los modelos son los mismos del experimento 02. Lo único nuevo es `selection_criterion='worst'` en `ExperimentConfig`. Los resultados se guardan en `outputs/experimentos/03_seleccion_robusta/`.

## 1. Entorno y datos

Resolvemos la raíz del repo (local o Colab) y aseguramos que los CSV de Olist están en `data/raw/`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('experimentos', 'notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Repositorio detectado:', root.name)

Repositorio detectado: LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile

raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))

CSV disponibles en data/raw: 9


## 2. Configuración y dataset enriquecido

Misma construcción que el experimento 02 —dataset a nivel de pedido, enriquecido con distancia cliente↔vendedor—, pero fijamos `selection_criterion='worst'`. Las demás constantes (semilla, partición, bootstrap) no cambian.

In [4]:
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.geography import attach_order_geography
from src.experiments import run_task

In [5]:
MAX_ROWS = None  # Use None para la corrida completa.
config = ExperimentConfig(
    max_rows=MAX_ROWS,
    cv_splits=3,
    bootstrap_iterations=500,
    selection_criterion='worst',
)
config

ExperimentConfig(random_state=42, max_rows=None, cv_splits=3, bootstrap_iterations=500, min_frequency=20, low_cardinality_max=30, high_cardinality_min=100, selection_criterion='worst')

In [6]:
tables = load_olist_tables(raw)
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
coverage = orders['mean_customer_seller_distance_km'].notna().mean()
print(f'Pedidos enriquecidos: {len(orders):,}, cobertura geográfica: {coverage:.2%}')

Pedidos enriquecidos: 99,441, cobertura geográfica: 98.73%


## 3. Regresión — días hasta la entrega

El ranking se ordena ahora por `cv_worst` (el peor pliegue), no por `validation_MAE`. Reportamos las dos columnas de CV (media y peor) para dejar ver cómo se mueve el reparto, no solo quién gana.

In [7]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
X_reg.shape

(96476, 24)

In [8]:
reg = run_task('regression', X_reg, y_reg, dates_reg, config)

/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [9]:
reg['ranking'][['model', 'cv_mean', 'cv_worst', 'validation_MAE', 'test_MAE', 'test_ci95_low', 'test_ci95_high']]

,model,cv_mean,cv_worst,validation_MAE,test_MAE,test_ci95_low,test_ci95_high
0,Random Forest (ensamble),-5.564920,-6.355599,5.389473,4.144124,4.087904,4.197696
1,Ridge (parametrico),-5.623899,-6.431760,5.538340,5.057653,4.997014,5.118160
2,KNN (no parametrico),-5.711153,-6.549184,5.577923,4.614408,4.560888,4.677054
3,MLP (red neuronal),-5.631504,-6.551321,5.327233,4.358905,4.297216,4.419671
4,LinearSVR (SVM),-5.651001,-6.777811,5.305392,3.893764,3.837104,3.950129
5,Baseline trivial (mediana),-6.840587,-7.761992,6.799813,4.962445,4.909492,5.020134


El criterio robusto cambia la familia campeón: ahora gana Random Forest porque su peor pliegue (−6.36) está por encima del de LinearSVR (−6.78). El cambio tiene costo medible: el RF seleccionado da MAE 4.14 en test, 0.22 días peor que el LinearSVR del experimento 02 (3.92). El criterio robusto convierte en castigo explícito la inestabilidad entre pliegues que el criterio por media ocultaba, pero para la métrica final de regresión en este dataset la media sigue siendo mejor elección.

## 4. Clasificación — probabilidad de entrega tardía

El objetivo real del experimento. En los experimentos 01 y 02 el campeón por media de CV (Random Forest) resultó peor que el baseline trivial sobre test, mientras que logística —descartada por la media— llega a 0.73 ROC-AUC en test. La hipótesis: si el criterio castiga la inestabilidad entre pliegues, Logística podría subir en el ranking.

In [10]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
X_cls.shape

(96476, 24)

In [11]:
cls = run_task('classification', X_cls, y_cls, dates_cls, config)

/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [12]:
cls['ranking'][['model', 'cv_mean', 'cv_worst', 'validation_ROC_AUC', 'test_ROC_AUC', 'test_ci95_low', 'test_ci95_high']]

,model,cv_mean,cv_worst,validation_ROC_AUC,test_ROC_AUC,test_ci95_low,test_ci95_high
0,Random Forest (ensamble),0.643319,0.584706,0.726523,0.636528,0.620895,0.652024
1,Logistica (parametrico),0.635790,0.566421,0.706497,0.727164,0.713872,0.740696
2,LinearSVC (SVM),0.633662,0.564917,0.704053,0.721014,0.708207,0.734688
3,KNN (no parametrico),0.594633,0.564740,0.653752,0.571053,0.553257,0.587355
4,MLP (red neuronal),0.602559,0.544707,0.694186,0.679481,0.664629,0.694353
5,Baseline trivial (clase mayoritaria),0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


**La familia ganadora no cambia:** Random Forest gana tanto por media (0.646) como por peor pliegue (0.585). Logística queda segunda en ambos criterios (0.636 / 0.566). Lo que sí cambia son los **hiperparámetros** elegidos dentro de la familia. La media selecciona `max_depth=12, min_samples_leaf=1` (test ROC-AUC 0.533); el peor pliegue selecciona `max_depth=24, min_samples_leaf=2` (test ROC-AUC 0.637). El salto de +0.10 ROC-AUC dentro de la misma familia es la evidencia más clara del efecto del criterio robusto en este dataset.

Aun así, Logística sigue muy por encima en test (0.727) tanto del RF-mean como del RF-worst. El criterio ayuda a elegir hiperparámetros más estables dentro de la familia ganadora, pero no resuelve el shift que favorece a la familia lineal: **el shift que importa está entre desarrollo y prueba, no entre pliegues de la CV**.

## 5. Comparación con los experimentos anteriores

Reunimos los tres `summary.json` para dejar en una tabla la secuencia completa.

In [13]:
import json
import pandas as pd

def read_summary(name):
    return json.loads((root / f'outputs/experimentos/{name}/summary.json').read_text())

exp01 = read_summary('01_baseline_pre_compra')
exp02 = read_summary('02_distancia_geografica')

comparison = pd.DataFrame(
    [
        {
            'experimento': '01 baseline',
            'regresión MAE': exp01['regression']['test_primary'],
            'clasificación ROC-AUC': exp01['classification']['test_primary'],
        },
        {
            'experimento': '02 + distancia',
            'regresión MAE': exp02['regression']['test_primary'],
            'clasificación ROC-AUC': exp02['classification']['test_primary'],
        },
        {
            'experimento': '03 criterio robusto',
            'regresión MAE': float(reg['ranking'].iloc[0]['test_MAE']),
            'clasificación ROC-AUC': float(cls['ranking'].iloc[0]['test_ROC_AUC']),
        },
    ]
)
comparison

,experimento,regresión MAE,clasificación ROC-AUC
0,01 baseline,3.979647,0.674767
1,02 + distancia,3.923850,0.532568
2,03 criterio robusto,4.144124,0.636528


## 6. Persistencia de resultados

Se guardan las mismas cuatro tablas por tarea y el `summary.json` que incluye el criterio empleado.

In [14]:
output = root / 'outputs' / 'experimentos' / '03_seleccion_robusta'
output.mkdir(parents=True, exist_ok=True)

for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)

def _headline(result, primary_col):
    top = result['ranking'].iloc[0]
    return {
        'champion': top['model'],
        'test_primary': float(top[primary_col]),
        'test_ci95': [float(top['test_ci95_low']), float(top['test_ci95_high'])],
    }

summary = {
    'sample_rows': MAX_ROWS,
    'cv_splits': config.cv_splits,
    'bootstrap_iterations': config.bootstrap_iterations,
    'selection_criterion': config.selection_criterion,
    'geo_coverage': float(coverage),
    'regression': _headline(reg, 'test_MAE'),
    'classification': _headline(cls, 'test_ROC_AUC'),
}
(output / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary

{'sample_rows': None,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'selection_criterion': 'worst',
 'geo_coverage': 0.9872889452036886,
 'regression': {'champion': 'Random Forest (ensamble)',
  'test_primary': 4.14412417440539,
  'test_ci95': [4.087903649734188, 4.197696003567892]},
 'classification': {'champion': 'Random Forest (ensamble)',
  'test_primary': 0.6365281750811616,
  'test_ci95': [0.6208947703957257, 0.6520242916618444]}}

## 7. Cierre

El experimento cumple su rol metodológico. Con el dataset completo y mallas ampliadas, el criterio robusto **sí** modifica decisiones:

- En regresión cambia la familia campeón (LinearSVR → Random Forest) con un costo medible: MAE 4.14 vs 3.92. La media sigue siendo mejor para la métrica final.
- En clasificación mantiene la familia (Random Forest) pero elige hiperparámetros más estables que generalizan +0.10 ROC-AUC en test (0.637 vs 0.533).
- Ninguno de los dos criterios cierra la brecha de shift que separa a Logística (0.727) del Random Forest (0.637 en el mejor caso). Esa brecha es estructural, entre desarrollo y prueba, y queda fuera del alcance del criterio de agregación de CV.

Con señales previas a la compra el techo queda:

- **Regresión:** LinearSVR del experimento 02, MAE 3.92 días en test.
- **Clasificación:** Random Forest por CV; Logística es el modelo que efectivamente generaliza a la ventana de test (ROC-AUC 0.727). La discrepancia se reporta como limitación por shift temporal.

Para mover más el resultado habría que acercar el holdout de validación al test en el tiempo o aplicar una corrección de shift explícita (reponderación por importancia, adaptación de dominio). Esto excede el alcance declarado del Entregable II y se deja como limitación en el informe.